In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
from workflow.scripts.extra_functions import get_bounding
import rasterio as rio
from rasterio.enums import Resampling
from numpy import empty

In [ ]:
method = snakemake.params.method

# input data
categories = list(snakemake.input)

# params
polygon_borders = snakemake.params.polygon_borders
target_crs = snakemake.params.target_crs # target EPSG

# output
output_path = snakemake.output.output_path

In [ ]:
# resolution
res = 100

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

In [ ]:

t=time.time()

if method=="manual":
    dst_transform, dst_shape = padded_transform_and_shape(
        bounds=rio.features.bounds(polygon_bounds), # cutout bounds
        res=res # resolution
    )

    raster_all = empty(dst_shape)

    for category in categories:
        src = rio.open(category) 
        raster_reprojected = empty(dst_shape)

        rio.warp.reproject(
            src.read(1),
            raster_reprojected,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
            num_threads=2
        )
        raster_all = raster_all + raster_reprojected
        print(time.time()-t)

        

    raster_all[raster_all>=1] = 1

    print(time.time()-t)

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_all.shape[1],
        'height': raster_all.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_all, 1)
    print(time.time()-t)


In [ ]:
t=time.time()

if method=="atlite":

    excluder_wind = ExclusionContainer(crs=target_crs)
    for category in categories:
        excluder_wind.add_raster(category,crs=target_crs)
    masked, transform = shape_availability(polygon_bounds.geometry, excluder_wind)

    mask_uint8 = (~masked).astype("uint8")

    # Prepare metadata
    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': mask_uint8.shape[1],
        'height': mask_uint8.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(mask_uint8, 1)
    print(time.time()-t)